In [1]:
import pandas as pd
import os


In [2]:
df = pd.read_csv("experiment_log.csv")
columns = {
    "deepseek-r1:1.5b": "DeepSeek-R1",
    "llama3.2:latest": "Llama3.2",
    "qwen2.5:latest": "Qwen2.5",
    "mistral:latest": "Mistral",
    "gemma3:1b": "Gemma3.1b",
    "assist2009": "ASSISTments 2009",
}
df["dataset_name"] = df["dataset_name"].apply(lambda x: columns[x])
df.rename(columns={"model_name": "Model", "dataset_name": "Dataset"}, inplace=True)
df

,Dataset,Model,trainauc,trainacc,validauc,validacc,testauc,testacc
0,Llama3.2,dkt,0.899642,0.868848,0.885480,0.851628,0.891975,0.856459
1,Mistral,dkt,0.852445,0.848602,0.840900,0.838405,0.837119,0.857436
2,Qwen2.5,dkt,0.877500,0.958880,0.892357,0.959918,0.854057,0.959547
3,DeepSeek-R1,dkt,0.810132,0.747142,0.782530,0.725490,0.774151,0.712366
4,ASSISTments 2009,dkt,0.865165,0.826422,0.798089,0.784864,0.794926,0.782129
5,Llama3.2,sakt,0.892414,0.869427,0.899306,0.846803,0.893287,0.851675
6,Mistral,sakt,0.887287,0.866281,0.852670,0.846800,0.825309,0.851282
7,Qwen2.5,sakt,0.936098,0.959774,0.878040,0.960946,0.869066,0.960356
8,DeepSeek-R1,sakt,0.802894,0.732924,0.785485,0.691176,0.770946,0.678763
9,ASSISTments 2009,sakt,0.843328,0.805354,0.653841,0.722693,0.666672,0.734738


In [3]:
df.sort_values(by=["Model", "Dataset"], inplace=True, ignore_index=True)
df

,Dataset,Model,trainauc,trainacc,validauc,validacc,testauc,testacc
0,ASSISTments 2009,akt,0.852197,0.813949,0.790921,0.784635,0.785136,0.782852
1,DeepSeek-R1,akt,0.788725,0.732088,0.777862,0.718137,0.780134,0.725806
2,Gemma3.1b,akt,0.823629,0.775443,0.818702,0.765443,0.827703,0.778391
3,Llama3.2,akt,0.880552,0.835843,0.885587,0.857660,0.885092,0.852632
4,Mistral,akt,0.830461,0.846030,0.829460,0.843652,0.818601,0.854359
5,Qwen2.5,akt,0.600629,0.958582,0.804714,0.959918,0.813705,0.960356
6,ASSISTments 2009,dkt,0.865165,0.826422,0.798089,0.784864,0.794926,0.782129
7,DeepSeek-R1,dkt,0.810132,0.747142,0.782530,0.725490,0.774151,0.712366
8,Gemma3.1b,dkt,0.829251,0.784557,0.809179,0.773500,0.824749,0.788644
9,Llama3.2,dkt,0.899642,0.868848,0.885480,0.851628,0.891975,0.856459


In [4]:
def create_pivot_table(
    df: pd.DataFrame, value_column: str, table_name: str
) -> pd.DataFrame:
    pt = pd.pivot_table(
        df,
        values=value_column,
        index="Dataset",
        columns="Model",
        margins=True,
        margins_name="Average",
    )

    column_format = "+l^c^c^c^c^c^>{\\bfseries}^c"
    caption = f"{table_name} across LLM Synthesizers and Knowledge Tracing Models."
    label = f"\\label{{tab:{value_column}}}"

    pt_latex: str = pt.to_latex(
        float_format="%.4f",
        column_format=column_format,
        index_names=False,
        header=["AKT", "DKT", "DTransformer", "SAKT", "SimpleKT", "Average"],
        caption=caption + label,
        position="H",
    )
    kt_title = "&\\multicolumn{6}{c}{\\textbf{Knowledge Tracing Algorithms}}\\\\ \n"
    bold_rowstyle = "\\rowstyle{\\bfseries}"

    pt_latex = pt_latex.replace("\nModel", "\nSynthesizer")
    pt_latex = pt_latex.replace("[H]", "[H]\n\\centering")
    pt_latex = pt_latex.replace("DeepSeek-R1", "\\midrule\nDeepSeek-R1")
    pt_latex = pt_latex.replace("\\toprule", "\\toprule\n" + kt_title + bold_rowstyle)
    pt_latex = pt_latex.replace("\nAverage", f"\n\\midrule\n{bold_rowstyle}\nAverage")

    os.makedirs("tables", exist_ok=True)
    with open(f"../dissertation/tables/{value_column}.tex", "w") as f:
        f.write(pt_latex)

    return pt

In [5]:
pivot_table_names = {
    "trainacc": "Last epoch \\textbf{Train Accuracy}",
    "trainauc": "Last epoch \\textbf{Train AUC}",
    "validacc": "Last epoch \\textbf{Validation Accuracy}",
    "validauc": "Last epoch \\textbf{Validation AUC}",
    "testacc": "\\textbf{Test Accuracy}",
    "testauc": "\\textbf{Test AUC}",
}

tables = {}
for value, table_name in pivot_table_names.items():
    tables[value] = create_pivot_table(df, value, table_name)

In [6]:
print(abs(tables["trainauc"] - tables["validauc"]).mean().mean())
print(abs(tables["trainauc"] - tables["testauc"]).mean().mean())
print(abs(tables["trainacc"] - tables["validacc"]).mean().mean())
print(abs(tables["trainacc"] - tables["testacc"]).mean().mean())

print()

print(abs(tables["trainauc"] - tables["validauc"]).T.mean().mean())
print(abs(tables["trainauc"] - tables["testauc"]).T.mean().mean())
print(abs(tables["trainacc"] - tables["validacc"]).T.mean().mean())
print(abs(tables["trainacc"] - tables["testacc"]).T.mean().mean())

0.04049482784379895
0.04362075537348456
0.024201767923875992
0.02358461340033871

0.04049482784379895
0.04362075537348455
0.024201767923875992
0.023584613400338707


In [7]:
pd.set_option("display.width", 88)
pd.set_option("display.max_columns", None)

print("trainauc - validauc")
print(abs(tables["trainauc"] - tables["validauc"]))
print()

print("trainauc - testauc")
print(abs(tables["trainauc"] - tables["testauc"]))
print()

print("trainacc - validacc")
print(abs(tables["trainacc"] - tables["validacc"]))
print()

print("trainacc - testacc")
print(abs(tables["trainacc"] - tables["testacc"]))


trainauc - validauc
Model                  akt       dkt  dtransformer      sakt  simplekt   Average
Dataset                                                                         
ASSISTments 2009  0.061276  0.067076      0.161428  0.189488  0.162182  0.128290
DeepSeek-R1       0.010863  0.027602      0.041331  0.017409  0.020084  0.023458
Gemma3.1b         0.004927  0.020072      0.003338  0.041526  0.029493  0.019871
Llama3.2          0.005035  0.014162      0.006414  0.006892  0.005478  0.002825
Mistral           0.001001  0.011544      0.023314  0.034617  0.008320  0.015759
Qwen2.5           0.204085  0.014857      0.001448  0.058058  0.023374  0.027792
Average           0.021842  0.020933      0.039063  0.055701  0.041488  0.027069

trainauc - testauc
Model                  akt       dkt  dtransformer      sakt  simplekt   Average
Dataset                                                                         
ASSISTments 2009  0.067061  0.070239      0.160277  0.176656  0.16625

In [9]:
print(abs(tables["testauc"] - tables["trainauc"]).T.mean())

Dataset
ASSISTments 2009    0.128098
DeepSeek-R1         0.027000
Gemma3.1b           0.021845
Llama3.2            0.003815
Mistral             0.028721
Qwen2.5             0.057427
Average             0.038441
dtype: float64


In [8]:
print(abs(tables["testacc"] - tables["trainacc"]).T.mean())


Dataset
ASSISTments 2009    0.067489
DeepSeek-R1         0.025973
Gemma3.1b           0.021896
Llama3.2            0.013234
Mistral             0.014203
Qwen2.5             0.001315
Average             0.020983
dtype: float64


In [11]:
print(abs(tables["testauc"] - tables["trainauc"]).mean())

Model
akt             0.047366
dkt             0.026193
dtransformer    0.044254
sakt            0.062462
simplekt        0.044636
Average         0.036814
dtype: float64


In [10]:
print(abs(tables["testacc"] - tables["trainacc"]).mean())

Model
akt             0.009782
dkt             0.016861
dtransformer    0.031854
sakt            0.031087
simplekt        0.030565
Average         0.021359
dtype: float64
